# Model 4: Satisfaction Prediction

Predict satisfaction scores for open support tickets from ticket metadata and description text. The model trains only on completed tickets with real ratings; the raw data is used because the processed file fills missing open-ticket ratings with a default value.

The workflow covers data inspection, metadata preparation, TF-IDF text features, a reproducible train/test split, a combined preprocessing pipeline, five-fold model comparison, held-out MAE/RMSE/R² evaluation, and separate exports for test predictions and open-ticket scores. Status and resolution time are excluded from predictors because they are unavailable or reveal post-resolution information for open tickets.

In [2]:
# Model 4: predict satisfaction scores for open support tickets.
# Learn from genuine ratings on completed tickets; predict Pending and Escalated tickets.
# Use the raw ticket file because the processed file fills missing open-ticket targets with 3.

# 1. Import Required Libraries
# Import Path so data folders can be found from the project root or notebooks folder.  # Explain paths.
from pathlib import Path
# Import find_spec so XGBoost is an optional candidate.  # Explain dependency check.
from importlib.util import find_spec
# Import NumPy and pandas for numeric work and ticket tables.  # Explain data tools.
import numpy as np
import pandas as pd
# Import scikit-learn tools for feature processing, regression, validation, and metrics.  # Explain ML tools.
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Fix random choices so split, cross-validation, and tree models are repeatable.  # Explain reproducibility.
RANDOM_STATE = 42
# Find the project directory based on the current notebook launch location.  # Explain project path.
PROJECT_DIR = Path.cwd()
# Move up one directory when this notebook is running from notebooks/.  # Explain path fallback.
if not (PROJECT_DIR / "data" / "raw").exists():
    PROJECT_DIR = PROJECT_DIR.parent
# Define the raw input file and the two result files.  # Explain file paths.
RAW_FILE = PROJECT_DIR / "data" / "raw" / "support_tickets (1).csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TEST_OUTPUT_FILE = PROCESSED_DIR / "output_customer_satisfaction_test_predictions.csv"
OPEN_OUTPUT_FILE = PROCESSED_DIR / "output_customer_satisfaction_predictions.csv"

# 2. Load and Inspect Data
# Load source tickets without trusting automatic date inference.  # Explain raw source.
tickets = pd.read_csv(RAW_FILE)
# Parse ticket dates explicitly because CSV date inference can leave them as strings.  # Explain date parsing.
tickets["ticket_date"] = pd.to_datetime(tickets["ticket_date"], errors="coerce")
# Convert the target to numbers while retaining genuinely missing scores.  # Explain target parsing.
tickets["customer_satisfaction_score"] = pd.to_numeric(tickets["customer_satisfaction_score"], errors="coerce")
# Display a small sample to inspect metadata, text, status, and target columns.  # Explain preview.
display(tickets.head())
# Count missing values and verify that open-ticket scores are absent in the raw source.  # Explain missingness check.
print("Missing values by column:")
print(tickets.isna().sum().to_string())
# Normalize status values so status selection ignores capitalization and surrounding spaces.  # Explain status cleanup.
tickets["status_normalized"] = tickets["ticket_status"].astype("string").str.strip().str.casefold()
# In this dataset, Closed and Resolved are completed statuses; Pending and Escalated are open.  # Explain status rules.
completed_statuses = {"closed", "resolved"}
open_statuses = {"pending", "escalated"}
# Keep completed tickets with real satisfaction scores as labeled training data.  # Explain training cohort.
scored_tickets = tickets.loc[
    tickets["status_normalized"].isin(completed_statuses)
    & tickets["customer_satisfaction_score"].notna()
].copy()
# Keep open tickets without real scores for prediction after evaluation.  # Explain open cohort.
open_tickets = tickets.loc[
    tickets["status_normalized"].isin(open_statuses)
    & tickets["customer_satisfaction_score"].isna()
].copy()
# Report the number of observed training labels and open tickets to score.  # Explain cohort sizes.
print(f"Completed tickets with observed ratings: {len(scored_tickets):,}")
print(f"Open tickets to score: {len(open_tickets):,}")
# Stop if the source data does not contain both groups needed by the workflow.  # Explain data validation.
if scored_tickets.empty or open_tickets.empty:
    raise ValueError("Need rated completed tickets and open tickets with missing scores.")

# 3. Preprocess Ticket Metadata
# Derive calendar metadata available when the ticket is created.  # Explain date features.
for ticket_frame in (scored_tickets, open_tickets):
    ticket_frame["ticket_month"] = ticket_frame["ticket_date"].dt.month
    ticket_frame["ticket_weekday"] = ticket_frame["ticket_date"].dt.dayofweek
# Choose metadata available before resolution, including category, priority, and assigned agent.  # Explain predictors.
categorical_columns = ["issue_category", "issue_subcategory", "priority", "support_agent_id"]
numeric_columns = ["ticket_month", "ticket_weekday"]
# Exclude ticket_status because open statuses never occur among labeled training rows.  # Explain status exclusion.
# Exclude resolution_time_hours because it is unknown for open tickets and only known after resolution.  # Explain leakage prevention.
# Replace missing descriptions with empty text so the vectorizer can process all tickets.  # Explain text missingness.
scored_tickets["ticket_description"] = scored_tickets["ticket_description"].fillna("").astype(str)
open_tickets["ticket_description"] = open_tickets["ticket_description"].fillna("").astype(str)
# Use only metadata and text; exclude ticket IDs, statuses, resolution time, and the target.  # Explain feature selection.
feature_columns = categorical_columns + numeric_columns + ["ticket_description"]
X = scored_tickets[feature_columns].copy()
y = scored_tickets["customer_satisfaction_score"].copy()
X_open = open_tickets[feature_columns].copy()
# Remember the observed score limits so predictions stay on the rating scale.  # Explain target range.
SCORE_MIN, SCORE_MAX = float(y.min()), float(y.max())
print(f"Observed satisfaction scale: {SCORE_MIN:g} to {SCORE_MAX:g}")

# 4. Extract Text Features from Ticket Descriptions
# Convert words and two-word phrases into TF-IDF features within each training fold.  # Explain text processing.
text_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=3000,
)
# Show an example description used by the text model.  # Explain text preview.
print("Example ticket description:")
print(scored_tickets["ticket_description"].iloc[0])

# 5. Combine Metadata and Text Features
# Impute missing categories and one-hot encode them, allowing new categories at prediction time.  # Explain category pipeline.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
# Impute and scale numeric calendar features without centering sparse matrices.  # Explain numeric pipeline.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])
# Apply text, categorical, and numeric processing and join them into one feature matrix.  # Explain feature combination.
preprocessor = ColumnTransformer([
    ("text", text_vectorizer, "ticket_description"),
    ("categorical", categorical_pipeline, categorical_columns),
    ("numeric", numeric_pipeline, numeric_columns),
], remainder="drop")

# 6. Split Data into Training and Test Sets
# Reserve 20 percent of completed, rated tickets for a final held-out test.  # Explain test split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
# Show the number of records in each split.  # Explain split sizes.
print(f"Training examples: {len(X_train):,}; held-out examples: {len(X_test):,}")

# 7. Build Preprocessing Pipelines for Metadata and Text
# Keep transformations inside each model pipeline so validation folds cannot leak vocabulary or imputations.  # Explain fold safety.

# 8. Define and Compare Regression Models
# Ridge is a regularized linear baseline suited to high-dimensional sparse TF-IDF features.  # Explain linear model.
candidates = {
    "Ridge Regression": Ridge(alpha=2.0, solver="lsqr"),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, max_features=0.5,
        random_state=RANDOM_STATE, n_jobs=1,
    ),
}
# Include XGBoost if available; otherwise continue with the other candidates.  # Explain optional model.
if find_spec("xgboost") is not None:
    from xgboost import XGBRegressor
    candidates["XGBoost Regressor"] = XGBRegressor(
        n_estimators=150, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=RANDOM_STATE,
        n_jobs=1, verbosity=0,
    )
else:
    print("XGBoost skipped because the xgboost package is not installed.")

# 9. Cross-Validate Candidate Models
# Use the same shuffled five-fold split for every candidate.  # Explain CV setup.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# Ask cross-validation to report MAE, RMSE, and R-squared for each fold.  # Explain CV metrics.
scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
# Store average scores and pipelines for later selection and fitting.  # Explain storage.
cv_rows, candidate_pipelines = [], {}
# Fit and score every candidate using only the training split.  # Explain comparison loop.
for model_name, estimator in candidates.items():
    # Attach identical text and metadata preprocessing to the current regressor.  # Explain model pipeline.
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", estimator),
    ])
    # Evaluate on five validation folds and raise any modeling error immediately.  # Explain CV execution.
    scores = cross_validate(
        model_pipeline, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, error_score="raise",
    )
    # Keep the unfitted pipeline so the selected model can be trained later.  # Explain pipeline storage.
    candidate_pipelines[model_name] = model_pipeline
    # Convert scikit-learn's negative error scores into positive rating errors.  # Explain score conversion.
    cv_rows.append({
        "model": model_name,
        "cv_mae": -scores["test_mae"].mean(),
        "cv_rmse": -scores["test_rmse"].mean(),
        "cv_r2": scores["test_r2"].mean(),
    })
# Rank candidates by lowest CV MAE, then use RMSE to break ties.  # Explain selection rule.
cv_results = pd.DataFrame(cv_rows).sort_values(["cv_mae", "cv_rmse"])
# Display the comparison scores used to select the model.  # Explain CV report.
print("Five-fold cross-validation results:")
print(cv_results.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

# 10. Select and Train Final Model
# Choose the candidate with the lowest cross-validated MAE.  # Explain selected model.
selected_name = cv_results.iloc[0]["model"]
selected_pipeline = candidate_pipelines[selected_name]
print(f"Selected model by CV MAE: {selected_name}")
# Fit this model on the training portion before evaluating the held-out portion.  # Explain evaluation fit.
test_model = clone(selected_pipeline).fit(X_train, y_train)

# 11. Evaluate Model on Held-Out Test Set
# Predict satisfaction ratings for tickets excluded from model fitting.  # Explain test predictions.
test_predictions = test_model.predict(X_test)
# Keep predictions within the observed satisfaction rating range.  # Explain score bounds.
test_predictions = np.clip(test_predictions, SCORE_MIN, SCORE_MAX)
# Calculate the requested regression metrics against held-out observed scores.  # Explain evaluation.
test_metrics = {
    "MAE": mean_absolute_error(y_test, test_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    "R2": r2_score(y_test, test_predictions),
}
# Print each held-out metric.  # Explain metric output.
print("Held-out test metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name:>8}: {metric_value:.3f}")
# Interpret errors in rating points on the observed score scale.  # Explain metric meaning.
print(
    f"MAE is the average absolute rating error ({test_metrics['MAE']:.2f} points); "
    f"RMSE penalizes larger mistakes more; R2 summarizes explained variation "
    f"(test R2={test_metrics['R2']:.3f})."
)

# 12. Export Held-Out Predictions
# Pair each test ticket's identifier with its real rating and predicted rating.  # Explain test export.
test_report = pd.DataFrame({
    "ticket_id": scored_tickets.loc[X_test.index, "ticket_id"].to_numpy(),
    "actual_customer_satisfaction_score": y_test.to_numpy(),
    "predicted_customer_satisfaction_score": test_predictions,
})
# Save held-out predictions separately for error analysis.  # Explain test file.
test_report.to_csv(TEST_OUTPUT_FILE, index=False)
# Refit the selected pipeline on all genuinely rated completed tickets.  # Explain final model fit.
final_model = clone(selected_pipeline).fit(X, y)
# Estimate the score of every open ticket that lacks a real target.  # Explain open-ticket prediction.
open_predictions = final_model.predict(X_open)
# Keep estimated ratings within the observed target scale.  # Explain open score bounds.
open_predictions = np.clip(open_predictions, SCORE_MIN, SCORE_MAX)
# Preserve useful ticket details beside each predicted satisfaction score.  # Explain open report.
open_report = open_tickets[[
    "ticket_id", "customer_id", "ticket_date", "issue_category",
    "issue_subcategory", "priority", "ticket_status",
]].copy()
# Attach the predicted rating to the corresponding open ticket.  # Explain prediction column.
open_report["predicted_customer_satisfaction_score"] = open_predictions
# Save open-ticket estimates as a separate file for support follow-up.  # Explain open output.
open_report.to_csv(OPEN_OUTPUT_FILE, index=False)
# Confirm the size and location of both generated prediction files.  # Explain completion.
print(f"Saved {len(test_report):,} held-out predictions to: {TEST_OUTPUT_FILE}")
print(f"Saved {len(open_report):,} open-ticket predictions to: {OPEN_OUTPUT_FILE}")

,ticket_id,customer_id,ticket_date,issue_category,issue_subcategory,priority,ticket_status,resolution_time_hours,customer_satisfaction_score,ticket_description,support_agent_id
0,TCK0000001,CUST000001,2024-12-05,Subscription,Auto-renew complaint,Medium,Closed,15.8,2.0,I was auto renewed without any reminder notifi...,AGT0050
1,TCK0000002,CUST000001,2025-02-06,Account Access,Login failure,Medium,Closed,6.0,3.0,I cannot log into my account from my phone eve...,AGT0120
2,TCK0000003,CUST000002,2026-06-15,Billing & Payments,Failed payment,Medium,Resolved,6.4,3.0,My payment failed twice this month even though...,AGT0012
3,TCK0000004,CUST000003,2021-03-05,Account Access,Account locked,Low,Escalated,NaN,NaN,My account got locked after multiple login att...,AGT0101
4,TCK0000005,CUST000007,2026-02-16,Account Access,Account locked,Medium,Resolved,2.4,3.0,My account got locked after multiple login att...,AGT0109


Missing values by column:
ticket_id                         0
customer_id                       0
ticket_date                      61
issue_category                    0
issue_subcategory                 0
priority                          0
ticket_status                     0
resolution_time_hours          1528
customer_satisfaction_score    1528
ticket_description                0
support_agent_id                  0
Completed tickets with observed ratings: 4,592
Open tickets to score: 1,528
Observed satisfaction scale: 1 to 5
Example ticket description:
I was auto renewed without any reminder notification and would like a refund for this cycle.
Training examples: 3,673; held-out examples: 919
Five-fold cross-validation results:
                  model  cv_mae  cv_rmse  cv_r2
      XGBoost Regressor   0.899    1.094  0.025
       Ridge Regression   0.909    1.106  0.003
Random Forest Regressor   0.917    1.117 -0.017
Selected model by CV MAE: XGBoost Regressor
Held-out test metrics:
 